# Text Preprocessing & Embeddings

From raw text to numerical representations that ML models can consume.

1. **Text Preprocessing** - Tokenization, stopwords, stemming, lemmatization
2. **Bag of Words & TF-IDF** - Classical sparse representations
3. **Word2Vec** - Dense word embeddings
4. **Sentence Embeddings** - From word-level to document-level representations

**Dataset**: 20 Newsgroups

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from collections import Counter
import re

sns.set_theme(style="whitegrid")

In [ ]:
categories = ["sci.med", "sci.space", "rec.sport.baseball", "talk.politics.guns"]
data = fetch_20newsgroups(subset="train", categories=categories,
                          remove=("headers", "footers", "quotes"))

texts = data.data
labels = data.target
label_names = data.target_names

print(f"Samples: {len(texts)}")
print(f"\nSample text:\n{texts[0][:300]}...")

## 1. Text Preprocessing Pipeline

In [ ]:
def preprocess_text(text):
    """Basic text preprocessing."""
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Remove non-alphabetic
    text = re.sub(r"\s+", " ", text).strip()  # Normalize whitespace
    return text

# Before and after
sample = texts[5]
print(f"Before: {sample[:200]}")
print(f"\nAfter: {preprocess_text(sample)[:200]}")

## 2. Bag of Words vs TF-IDF

| Method | Representation | Key Idea |
|--------|---------------|----------|
| **Bag of Words** | Word counts | Simple frequency |
| **TF-IDF** | Term frequency × inverse document frequency | Downweights common words |

$$\text{TF-IDF}(t,d) = \text{TF}(t,d) \times \log\frac{N}{\text{DF}(t)}$$

In [ ]:
# Compare BoW vs TF-IDF for classification
bow_pipe = Pipeline([
    ("vectorizer", CountVectorizer(max_features=10000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000)),
])

tfidf_pipe = Pipeline([
    ("vectorizer", TfidfVectorizer(max_features=10000, stop_words="english", ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000)),
])

bow_scores = cross_val_score(bow_pipe, texts, labels, cv=5, scoring="accuracy")
tfidf_scores = cross_val_score(tfidf_pipe, texts, labels, cv=5, scoring="accuracy")

print(f"Bag of Words:  {bow_scores.mean():.3f} (+/- {bow_scores.std():.3f})")
print(f"TF-IDF:        {tfidf_scores.mean():.3f} (+/- {tfidf_scores.std():.3f})")

In [ ]:
# Visualize TF-IDF: top terms per class
tfidf = TfidfVectorizer(max_features=10000, stop_words="english")
X_tfidf = tfidf.fit_transform(texts)
feature_names = tfidf.get_feature_names_out()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, class_idx in zip(axes.flat, range(len(label_names))):
    class_mask = labels == class_idx
    mean_tfidf = X_tfidf[class_mask].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-15:]
    
    ax.barh(range(15), mean_tfidf[top_idx], color="teal")
    ax.set_yticks(range(15))
    ax.set_yticklabels(feature_names[top_idx])
    ax.set_title(f"{label_names[class_idx]}")

plt.suptitle("Top TF-IDF Terms per Category", fontsize=14)
plt.tight_layout()
plt.show()

## 3. From Sparse to Dense: SVD on TF-IDF

Latent Semantic Analysis (LSA) applies SVD to reduce the TF-IDF matrix to dense, lower-dimensional representations.

In [ ]:
# LSA: TF-IDF + SVD
lsa_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, stop_words="english")),
    ("svd", TruncatedSVD(n_components=100)),
    ("clf", LogisticRegression(max_iter=1000)),
])

lsa_scores = cross_val_score(lsa_pipe, texts, labels, cv=5, scoring="accuracy")
print(f"LSA (100d):    {lsa_scores.mean():.3f} (+/- {lsa_scores.std():.3f})")

# Visualize in 2D
svd_2d = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=10000, stop_words="english")),
    ("svd", TruncatedSVD(n_components=2)),
])
X_2d = svd_2d.fit_transform(texts)

plt.figure(figsize=(8, 6))
for i, name in enumerate(label_names):
    mask = labels == i
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1], label=name, s=10, alpha=0.5)
plt.legend()
plt.title("LSA: Documents in 2D")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.tight_layout()
plt.show()

## 4. Word Embeddings with Gensim (Optional)

Word2Vec learns dense vector representations where semantically similar words are close.

```python
# pip install gensim
from gensim.models import Word2Vec

# Tokenize
tokenized = [preprocess_text(t).split() for t in texts]

# Train Word2Vec
w2v = Word2Vec(tokenized, vector_size=100, window=5, min_count=5, workers=4)

# Similar words
print(w2v.wv.most_similar("space", topn=5))

# Document embedding = mean of word vectors
def doc_embedding(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)
```

## Evolution of Text Representations

| Era | Method | Dimensions | Context-Aware |
|-----|--------|-----------|---------------|
| Classical | BoW / TF-IDF | 10k-100k (sparse) | No |
| Embedding | Word2Vec / GloVe | 100-300 (dense) | No (static) |
| Pre-trained | ELMo | 1024 | Yes (contextualized) |
| Transformer | BERT / GPT | 768-1024 | Yes (deep bidirectional) |

## Key Takeaways

1. **TF-IDF + logistic regression** is a surprisingly strong baseline for text classification
2. **n-grams** (bigrams, trigrams) capture local word order that BoW misses
3. **LSA/SVD** reduces dimensionality and captures latent semantic structure
4. **Word2Vec** learns that king - man + woman ≈ queen
5. **For production NLP, use pre-trained transformers** (next notebook) - they encode context